<a href="https://colab.research.google.com/github/Tifym7/AquaGraph/blob/carla/Cassini2026_S1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:

!pip install -q earthengine-api geemap geopandas folium matplotlib numpy

In [39]:

import ee
import geemap
import json
import os
from datetime import datetime

In [40]:

PROJECT_ID = "cassini2026"

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized.")

ROMANIA = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "Romania"))
    .geometry()
)

AOI = ROMANIA
CENTER = [45.9, 24.9]
ZOOM = 7

EVENT_START = "2024-06-01"
EVENT_END = "2024-06-30"
BASELINE_START = "2019-06-01"
BASELINE_END = "2023-08-31"

Earth Engine initialized.


In [41]:


ORBIT_PASS = None
WATER_OCCURRENCE_MIN = 20
RIVER_BUFFER_M = 120
SMOOTHING_RADIUS_M = 30
OIL_PROBABILITY_THRESHOLD = 0.55

RIVERS_ASSET = "projects/cassini2026/assets/eu-hydro"


def event_month_filter(start_date, end_date):
    start_month = datetime.strptime(start_date, "%Y-%m-%d").month
    end_month = datetime.strptime(end_date, "%Y-%m-%d").month

    if start_month <= end_month:
        months = list(range(start_month, end_month + 1))
    else:
        months = list(range(start_month, 13)) + list(range(1, end_month + 1))

    filters = [ee.Filter.calendarRange(month, month, "month") for month in months]

    if len(filters) == 1:
        return filters[0]

    return ee.Filter.Or(*filters)

def mask_s1_edges(img):
    angle = img.select("angle")
    mask = angle.gt(30).And(angle.lt(45))
    return img.updateMask(mask).select(["VV", "VH"]).copyProperties(img, img.propertyNames())

def get_s1_collection(region, start_date, end_date):
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    )

    if ORBIT_PASS is not None:
        collection = collection.filter(ee.Filter.eq("orbitProperties_pass", ORBIT_PASS))

    return collection.map(mask_s1_edges)

def make_s1_composite(collection):
    return collection.median().focal_median(radius=SMOOTHING_RADIUS_M, units="meters")

def unit_scale_clamped(img, low, high):
    return img.subtract(low).divide(high - low).clamp(0, 1)

event_collection = get_s1_collection(AOI, EVENT_START, EVENT_END)
baseline_collection = get_s1_collection(AOI, BASELINE_START, BASELINE_END).filter(event_month_filter(EVENT_START, EVENT_END))

print("Event Sentinel-1 images:", event_collection.size().getInfo())
print("Baseline Sentinel-1 images:", baseline_collection.size().getInfo())

event_s1 = make_s1_composite(event_collection).clip(AOI)
baseline_s1 = make_s1_composite(baseline_collection).clip(AOI)

water_mask = (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select("occurrence")
    .gte(WATER_OCCURRENCE_MIN)
    .clip(AOI)
)


Event Sentinel-1 images: 97
Baseline Sentinel-1 images: 804


In [42]:

vv_event = event_s1.select("VV").rename("VV_EVENT")
vh_event = event_s1.select("VH").rename("VH_EVENT")
vv_baseline = baseline_s1.select("VV").rename("VV_BASELINE")
vh_baseline = baseline_s1.select("VH").rename("VH_BASELINE")
vv_darkening = vv_baseline.subtract(vv_event).rename("VV_DARKENING_DB")
vh_darkening = vh_baseline.subtract(vh_event).rename("VH_DARKENING_DB")

vv_texture = (
    event_s1.select("VV")
    .reduceNeighborhood(
        reducer=ee.Reducer.stdDev(),
        kernel=ee.Kernel.circle(radius=SMOOTHING_RADIUS_M, units="meters")
    )
    .rename("VV_TEXTURE")
)

vv_drop_score = unit_scale_clamped(vv_darkening, 1.5, 5.5)
vh_drop_score = unit_scale_clamped(vh_darkening, 1.0, 4.5)
absolute_dark_score = unit_scale_clamped(ee.Image.constant(-16).subtract(vv_event), 0, 8)
texture_score = unit_scale_clamped(ee.Image.constant(1.8).subtract(vv_texture), 0, 1.8)

oil_probability = (
    vv_drop_score.multiply(0.45)
    .add(vh_drop_score.multiply(0.25))
    .add(absolute_dark_score.multiply(0.20))
    .add(texture_score.multiply(0.10))
    .rename("OIL_PROBABILITY")
    .updateMask(water_mask)
)

dark_pixel = (
    oil_probability
    .gte(OIL_PROBABILITY_THRESHOLD)
    .rename("DARK_PIXEL")
    .updateMask(water_mask)
)

water_pixel = water_mask.rename("WATER_PIXEL")

oil_stack = ee.Image.cat([
    vv_event,
    vh_event,
    vv_baseline,
    vh_baseline,
    vv_darkening,
    vh_darkening,
    vv_texture,
    oil_probability,
    dark_pixel,
    water_pixel
])

In [43]:
import time

STATS_SCALE = 30
CHUNK_SIZE = 100
DRIVE_FOLDER = "CassiniProject"
EXPORT_PREFIX = "sentinel1_oil_rivers_part"

STAT_PROPERTIES = [
    "VV_EVENT",
    "VH_EVENT",
    "VV_BASELINE",
    "VH_BASELINE",
    "VV_DARKENING_DB",
    "VH_DARKENING_DB",
    "VV_TEXTURE",
    "OIL_PROBABILITY",
    "DARK_PIXEL",
    "WATER_PIXEL"
]

def safe_number(feature, name):
    val = feature.get(name)
    return ee.Number(
        ee.Algorithms.If(
            ee.Algorithms.IsEqual(val, None),
            0,
            val
        )
    )

def add_join_id(feature):
    return feature.set("join_id", feature.id())

def make_buffer_feature(feature):
    return (
        ee.Feature(feature.geometry().buffer(RIVER_BUFFER_M))
        .copyProperties(feature)
        .set("join_id", feature.get("join_id"))
    )

def attach_stats_to_line(feature):
    stats_feature = ee.Feature(feature.get("stats_feature"))
    stats_dict = stats_feature.toDictionary(STAT_PROPERTIES)
    return feature.set(stats_dict)

def compute_risk(feature):
    oil_probability_mean = safe_number(feature, "OIL_PROBABILITY")
    dark_fraction = safe_number(feature, "DARK_PIXEL")
    vv_darkening_mean = safe_number(feature, "VV_DARKENING_DB")
    water_fraction = safe_number(feature, "WATER_PIXEL")

    risk_score = (
        oil_probability_mean.multiply(70)
        .add(dark_fraction.multiply(30))
        .min(100)
        .max(0)
    )

    high_condition = risk_score.gte(45).Or(
        dark_fraction.gte(0.25).And(vv_darkening_mean.gte(3))
    )

    medium_condition = risk_score.gte(20).Or(
        dark_fraction.gte(0.10).And(vv_darkening_mean.gte(2))
    )

    risk_level = ee.Algorithms.If(
        water_fraction.lt(0.02),
        "LOW",
        ee.Algorithms.If(
            high_condition,
            "HIGH",
            ee.Algorithms.If(medium_condition, "MEDIUM", "LOW")
        )
    )

    return feature.set({
        "oil_probability": oil_probability_mean,
        "dark_fraction": dark_fraction,
        "vv_darkening_mean": vv_darkening_mean,
        "water_fraction": water_fraction,
        "risk_score": risk_score,
        "risk_level": risk_level
    })

def style_by_risk(feature):
    risk = ee.String(feature.get("risk_level"))

    color = ee.Algorithms.If(
        risk.compareTo("HIGH").eq(0),
        "red",
        ee.Algorithms.If(
            risk.compareTo("MEDIUM").eq(0),
            "yellow",
            "green"
        )
    )

    width = ee.Algorithms.If(
        risk.compareTo("HIGH").eq(0),
        4,
        ee.Algorithms.If(
            risk.compareTo("MEDIUM").eq(0),
            3,
            2
        )
    )

    return feature.set({
        "style": {
            "color": color,
            "width": width
        }
    })

def remove_internal_properties(feature):
    clean_dict = feature.toDictionary().remove(["stats_feature"])
    return ee.Feature(feature.geometry(), clean_dict)

def process_river_chunk(chunk):
    lines = chunk.map(add_join_id)
    buffers = lines.map(make_buffer_feature)

    stats_buffers = oil_stack.reduceRegions(
        collection=buffers,
        reducer=ee.Reducer.mean(),
        scale=STATS_SCALE,
        tileScale=4,
        maxPixelsPerRegion=1e8
    )

    joined = ee.Join.saveFirst("stats_feature").apply(
        primary=lines,
        secondary=stats_buffers,
        condition=ee.Filter.equals(
            leftField="join_id",
            rightField="join_id"
        )
    )

    return (
        ee.FeatureCollection(joined)
        .map(attach_stats_to_line)
        .map(compute_risk)
        .map(style_by_risk)
        .map(remove_internal_properties)
    )

rivers = ee.FeatureCollection(RIVERS_ASSET).filterBounds(AOI)
preview_chunk = ee.FeatureCollection(rivers.toList(10, 0))
preview_final = process_river_chunk(preview_chunk)

preview_task = ee.batch.Export.table.toDrive(
    collection=preview_final,
    description="sentinel1_oil_preview_10",
    folder=DRIVE_FOLDER,
    fileNamePrefix="sentinel1_oil_preview_10",
    fileFormat="GeoJSON"
)

preview_task.start()

print("Preview export started.")

Preview export started.


In [44]:
Map = geemap.Map(center=CENTER, zoom=ZOOM)

Map.addLayer(
    oil_probability,
    {"min": 0, "max": 1, "palette": ["white", "yellow", "orange", "red"]},
    "Oil Slick Probability"
)

Map.addLayer(
    dark_pixel.selfMask(),
    {"palette": ["red"]},
    "Oil Slick Candidates"
)

Map.addLayer(
    preview_final.style(**{"styleProperty": "style"}),
    {},
    "Preview River Oil Pollution Risk"
)

Map

Map(center=[45.9, 24.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [45]:
count = rivers.size().getInfo()
print("Total river segments:", count)

for start in range(0, count, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, count)
    print(f"Starting export {start} to {end}")

    chunk = ee.FeatureCollection(rivers.toList(CHUNK_SIZE, start))
    result = process_river_chunk(chunk)

    task = ee.batch.Export.table.toDrive(
        collection=result,
        description=f"{EXPORT_PREFIX}_{start}",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"{EXPORT_PREFIX}_{start}",
        fileFormat="GeoJSON"
    )

    task.start()
    time.sleep(2)

print("Export tasks started.")

Total river segments: 34701
Starting export 0 to 100
Starting export 100 to 200
Starting export 200 to 300
Starting export 300 to 400
Starting export 400 to 500
Starting export 500 to 600
Starting export 600 to 700
Starting export 700 to 800
Starting export 800 to 900
Starting export 900 to 1000
Starting export 1000 to 1100
Starting export 1100 to 1200
Starting export 1200 to 1300
Starting export 1300 to 1400
Starting export 1400 to 1500
Starting export 1500 to 1600
Starting export 1600 to 1700
Starting export 1700 to 1800
Starting export 1800 to 1900
Starting export 1900 to 2000
Starting export 2000 to 2100
Starting export 2100 to 2200
Starting export 2200 to 2300
Starting export 2300 to 2400
Starting export 2400 to 2500
Starting export 2500 to 2600
Starting export 2600 to 2700
Starting export 2700 to 2800
Starting export 2800 to 2900
Starting export 2900 to 3000
Starting export 3000 to 3100
Starting export 3100 to 3200
Starting export 3200 to 3300
Starting export 3300 to 3400
Startin